### Overview

As we went through Feature Extraction & Engineering, it was discovered that it would be best to Model and Create Player (Defender) Archetypes around the `Ball-Playing` mould.

For defenders to take part in build-up play and effectively function and aid in attack not only in the phase of "Rest Defense" but also in weaving passes and making calibrated runs is a commonly demanded aspect by managers in the modern game.

So, we can narrow our database (limited in itself, of course, as this is a solo run born of a fan's passion rather than armed with the full resources of an official football club) down to defenders' **Progressive Contributions**.

In [77]:
# Getting our Data in!

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

df1 = pd.read_csv('master_df.csv')
df1.rename(columns = {'Unnamed: 0': 'match_id'}, inplace = True)

In [78]:
# Ensuring that the Needed Base Columns for Important Transformations are in their Appropriate Date Type Forms

# Convert string representation of lists to actual lists/tuples

import ast

def parse_location(loc_str):
    if isinstance(loc_str, str):
        try:
            return ast.literal_eval(loc_str)
        except (ValueError, SyntaxError):
            return np.nan # Handle malformed strings
    return loc_str # Return as is if not a string (e.g., already np.nan or an actual list)

# Apply to 'location'
df1['location'] = df1['location'].apply(parse_location)

# Apply to 'pass_end_location' if it exists
if 'pass_end_location' in df1.columns:
    df1['pass_end_location'] = df1['pass_end_location'].apply(parse_location)
    print("Converted 'pass_end_location' column from string to list/tuple objects.")
else:
    print("'pass_end_location' column not found in DataFrame.")

# Apply to 'carry_end_location' if it exists
if 'carry_end_location' in df1.columns:
    df1['carry_end_location'] = df1['carry_end_location'].apply(parse_location)
    print("Converted 'carry_end_location' column from string to list/tuple objects.")
else:
    print("'carry_end_location' column not found in DataFrame.")

print("Converted 'location' column from string to list/tuple objects.")

Converted 'pass_end_location' column from string to list/tuple objects.
Converted 'carry_end_location' column from string to list/tuple objects.
Converted 'location' column from string to list/tuple objects.


In [79]:
# print("Overview of the Dataset:")
# df1.info()
# print("")
# print("Glimpse of the Dataset:")
# df1.sample(7)

## Advanced Player Metrics: Defensive Efforts Proxy Measure

### Subtask:
Calculate a proxy measure for player defensive efforts, combining 'Ball Recovery' events with 'Block' events that occur during 'Regular Play', 'Free Kick', or 'From Counter' play patterns.

**Reasoning**:
To create a 'Defensive efforts' proxy measure, I will first filter the `original_df` for events that are either 'Ball Recovery' or 'Block' events with specific `play_pattern` values ('Regular Play', 'Free Kick', 'From Counter'). Then, I will group these events by `match_id`, `player`, `team`, and `play_pattern` to count the occurrences, providing insights into a player's defensive tendencies across different game states.

In [80]:
# Identify relevant play patterns for Blocks
block_play_patterns = ['Regular Play', 'Free Kick', 'From Counter']

# Filter for 'Ball Recovery' events
ball_recovery_events = df1[df1['type'] == 'Ball Recovery'].copy()

# Filter for 'Block' events with specified play patterns
block_events = df1[
    (df1['type'] == 'Block') &
    (df1['play_pattern'].isin(block_play_patterns))
].copy()

# Combine the filtered events
defensive_proxy_events = pd.concat([ball_recovery_events, block_events])

# Group by player, match, team, and play_pattern to count defensive efforts
player_defensive_efforts = defensive_proxy_events.groupby(['match_id', 'player', 'team', 'play_pattern']).size().reset_index(name = 'defensive_effort_count')

# print("Player Defensive Efforts Proxy Measure Calculation Complete. Sample of results:")
# display(player_defensive_efforts.head())

## Standardize Timestamp

### Subtask:
Convert the `timestamp` column to a standardized format, representing minutes with two decimal places, between 0 and 90 minutes.

In [81]:
# 1. Function to standardize timestamp to minutes with two decimal places
def standardize_timestamp(ts):
    if pd.isna(ts):
        return np.nan
    try:
        # Use pd.to_timedelta for robust parsing of HH:MM:SS.mmm format
        td = pd.to_timedelta(ts)
        total_seconds = td.total_seconds()
        total_minutes = total_seconds / 60
        return round(total_minutes, 2)
    except ValueError:
        return np.nan # Handle malformed or unparseable time strings

# Apply the initial conversion to minutes (keep full precision for scaling)
df1['minutes_raw'] = df1['timestamp'].apply(standardize_timestamp)

# 2. Min-Max Scale to a 0-90 range (Typical of a Football match)
min_val = df1['minutes_raw'].min()
max_val = df1['minutes_raw'].max()

if max_val != min_val:
    # Formula: ((X - min) / (max - min)) * (target_max - target_min) + target_min
    df1['standardized_timestamp'] = ((df1['minutes_raw'] - min_val) / (max_val - min_val)) * 90
else:
    df1['standardized_timestamp'] = 0.0  # Handle edge case where all times are identical

# 3. Round to 2 decimal places at the very end
df1['standardized_timestamp'] = df1['standardized_timestamp'].round(2)

# Drop the temporary column if you don't need it
df1.drop(columns=['minutes_raw'], inplace = True)

# print("Standardized timestamp column created. Sample:")
# display(original_df[['timestamp', 'standardized_timestamp']].sample(6))

## Player Ball Possession: Time in Ball Receipt or Carry Events

### Subtask:
Calculate the time of match in which a player has the ball the most, based on 'Ball Receipt' and 'Carry' event types.

In [82]:
# Filter for 'Ball Receipt' and 'Carry' events
ball_possession_events = df1[
    df1['type'].isin(['Ball Receipt', 'Carry'])
].copy()

# Group by player, match, and team to count these events
player_ball_possession_counts = ball_possession_events.groupby(['match_id', 'player', 'team', 'standardized_timestamp']).size().reset_index(name = 'ball_possession_event_count')

# print("Player Ball Possession Event Counts. Sample of results:")
# display(player_ball_possession_counts.sample(4))

## Player Passing Characteristics: Average Pass Length and Most Frequent Pass Type

### Subtask:
Calculate the average pass length and the most frequent pass type for each player.

In [83]:
# Filter for 'Pass' events
pass_events = df1[df1['type'] == 'Pass'].copy()

# Calculate average pass length and most frequent pass type per player per match
player_passing_stats = pass_events.groupby(['match_id', 'player', 'team']).agg(
    avg_pass_length = ('pass_length', 'mean'),
    most_frequent_pass_style = ('pass_body_part', lambda x: x.mode()[0] if not x.mode().empty else np.nan) # Handle cases with no mode
).reset_index()

# Using the Passing Body Parts most used, we can also infer each Player's Preferred Foot

# print("Player Passing Characteristics. Sample of results:")
# display(player_passing_stats.sample(6))



---



## Consolidate All Advanced Metrics into the Main DataFrame (`original_df`)

### Subtask:
Merge `player_defensive_efforts`, refined `player_ball_possession_counts`, and `player_passing_stats` into the `original_df` to create a single master DataFrame suitable for subsequent analyses like PCA and Hierarchical Clustering. Unapplicable metrics will be `NaN`.

**Reasoning**:
To prepare for downstream analysis like PCA and clustering, all calculated advanced metrics need to be integrated into a single DataFrame. I will merge `player_defensive_efforts` and `player_passing_stats` into `original_df`. Then, I will recalculate `player_ball_possession_counts` to represent the total count of ball possession events per player per match and merge that into `original_df` as well. This ensures all relevant player-centric features are available in one place.

In [84]:
# Merge 'player_defensive_efforts' into original_df
# Remove potentially conflicting columns from original_df before merging to avoid MergeError on re-run
for col in ['defensive_effort_count', 'defensive_effort_count_x', 'defensive_effort_count_y']:
    if col in df1.columns:
        df1.drop(columns=[col], inplace=True)

original_df = pd.merge(df1, player_defensive_efforts,
                       on = ['match_id', 'player', 'team', 'play_pattern'],
                       how='left')

# print("Merged player_defensive_efforts. Sample with new column:")
# display(original_df[['match_id', 'player', 'team', 'play_pattern', 'defensive_effort_count']].sample(5))

# Merge 'player_passing_stats' into original_df
# Remove potentially conflicting columns from original_df before merging
for col in ['avg_pass_length', 'most_frequent_pass_style', 'avg_pass_length_x', 'most_frequent_pass_style_x', 'avg_pass_length_y', 'most_frequent_pass_style_y']:
    if col in original_df.columns:
        original_df.drop(columns=[col], inplace=True)

original_df = pd.merge(original_df, player_passing_stats,
                       on = ['match_id', 'player', 'team'],
                       how = 'left')

# print("\nMerged player_passing_stats. Sample with new columns:")
# display(original_df[['match_id', 'player', 'team', 'avg_pass_length', 'most_frequent_pass_style']].sample(5))

# Recalculate player_ball_possession_counts to get total events per match per player
# Filter for 'Ball Receipt' and 'Carry' events
ball_possession_events_filtered = original_df[
    original_df['type'].isin(['Ball Receipt', 'Carry'])
].copy()

# DIAGNOSTIC: Check for duplicates in grouping keys before aggregation
num_rows_filtered = len(ball_possession_events_filtered)
num_unique_groups = ball_possession_events_filtered.groupby(['match_id', 'player', 'team']).ngroups
print(f"\nDiagnostic for Ball Possession Events:")
print(f"Number of rows in ball_possession_events_filtered: {num_rows_filtered}")
print(f"Number of unique (match_id, player, team) groups: {num_unique_groups}")
if num_rows_filtered == num_unique_groups:
    print("WARNING: Every row in ball_possession_events_filtered is a unique (match_id, player, team) combination. This will naturally result in event counts of 1.")
else:
    print("INFO: There are multiple events per (match_id, player, team) combination, counts should be > 1 for some groups.")

# Group by player, match, and team to count these events (total per match), ignoring timestamp for aggregation
player_ball_possession_total_counts = ball_possession_events_filtered.groupby(['match_id', 'player', 'team']).size().reset_index(name = 'total_ball_possession_events_per_match')

# Display the calculated counts before merging
print("\nPlayer Ball Possession Total Counts (before merge):")
display(player_ball_possession_total_counts.sample(5))

# Merge the recalculated 'player_ball_possession_total_counts' into original_df
# Remove potentially conflicting columns from original_df before merging
for col in ['total_ball_possession_events_per_match', 'total_ball_possession_events_per_match_x', 'total_ball_possession_events_per_match_y']:
    if col in original_df.columns:
        original_df.drop(columns = [col], inplace = True)

original_df = pd.merge(original_df, player_ball_possession_total_counts,
                       on = ['match_id', 'player', 'team'],
                       how = 'left')

# print("\nMerged total_ball_possession_events_per_match. Sample with new column:")
# display(original_df[['match_id', 'player', 'team', 'total_ball_possession_events_per_match']].sample(5))

print("\nFinal original_df with all new metrics (first 5 rows and info):")

display(original_df.sample(6))
# print("")
# original_df.info()


Diagnostic for Ball Possession Events:
Number of rows in ball_possession_events_filtered: 28964
Number of unique (match_id, player, team) groups: 28964

Player Ball Possession Total Counts (before merge):


,match_id,player,team,total_ball_possession_events_per_match
14144,230011,Chris Smalling,Manchester United,1
11592,223390,Alberto Moreno Pérez,Liverpool,1
26921,266384,Mamadou Sakho,Liverpool,1
611,55510,Ousmane Dembélé,Barcelona,1
26268,264377,Per Mertesacker,Arsenal,1



Final original_df with all new metrics (first 5 rows and info):


,match_id,type,player,position,team,league_name,season_name,possession_team,possession,timestamp,under_pressure,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,duel_type,duration,duel_outcome,foul_committed_type,foul_committed_card,foul_committed_penalty,interception_outcome,minute,miscontrol_aerial_won,out,location,pass_body_part,pass_height,pass_angle,pass_inswinging,pass_outswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_aerial_won,pass_recipient,pass_straight,pass_switch,pass_technique,pass_type,play_pattern,player_off_permanent,standardized_timestamp,defensive_effort_count,avg_pass_length,most_frequent_pass_style,total_ball_possession_events_per_match
43667,104709,Pass,Nathaniel Edwin Clyne,Right Back,Liverpool,Premier League,2015/2016,Liverpool,26,00:10:10.176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.544460,NaN,NaN,NaN,NaN,NaN,10,NaN,NaN,"[70.3, 71.7]",Right Foot,Ground Pass,-2.400610,NaN,NaN,12.740487,NaN,NaN,NaN,NaN,NaN,NaN,"[60.9, 63.1]",NaN,Lucas Pezzini Leiva,NaN,NaN,NaN,Free Kick,From Free Kick,NaN,16.99,NaN,12.740487,Right Foot,NaN
22260,51721,Ball Receipt*,Clément Lenglet,Left Center Back,Barcelona,La Liga,2020/2021,Barcelona,118,00:19:49.630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64,NaN,NaN,"[29.4, 48.7]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Play,NaN,33.12,NaN,NaN,NaN,NaN
108098,288735,Block,Laurent Koscielny,Left Center Back,Arsenal,Premier League,2015/2016,Southampton,226,00:47:41.577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,92,NaN,NaN,"[9.0, 30.7]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,From Throw In,NaN,79.66,NaN,NaN,NaN,NaN
85969,224434,Carry,Daley Blind,Left Center Back,Manchester United,Premier League,2015/2016,Manchester United,76,00:38:26.937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[107.7, 4.8]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.960000,NaN,NaN,NaN,NaN,NaN,38,NaN,NaN,"[107.7, 4.8]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,From Corner,NaN,64.23,NaN,NaN,NaN,1.0
29456,68125,Carry,Sergino Dest,Right Wing Back,Barcelona,La Liga,2020/2021,Barcelona,18,00:12:05.803,True,NaN,NaN,NaN,NaN,NaN,NaN,"[105.9, 56.6]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.888062,NaN,NaN,NaN,NaN,NaN,12,NaN,NaN,"[105.9, 57.9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Play,NaN,20.21,NaN,NaN,NaN,1.0
57802,139630,Pass,Faustino Marcos Alberto Rojo,Left Center Back,Manchester United,Premier League,2015/2016,Manchester United,118,00:07:20.543,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.524258,NaN,NaN,NaN,NaN,NaN,52,NaN,NaN,"[42.7, 10.0]",Right Foot,Ground Pass,-0.281479,NaN,NaN,25.920069,NaN,NaN,NaN,NaN,NaN,NaN,"[67.6, 2.8]",NaN,Memphis Depay,NaN,NaN,NaN,NaN,From Free Kick,NaN,12.26,NaN,25.920069,Right Foot,NaN




---



In [85]:
# Feature Extraction

# Being a Data Scientist/Researcher at a Football Club
# Like Ian Graham was at Liverpool, I Dream of being at Manchester United!

def process_and_compress_spatial_events(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters for key on-ball movement events, drops NaN coordinate rows,
    calculates forward progression deltas, Euclidean distances, and progressive action flags.
    Returns a lightweight, memory-efficient spatial DataFrame.
    """
    # 1. Filter for core spatial movement types
    events = df[df['type'].isin(['Pass', 'Carry', 'Dribble'])].copy()

    # 2. Extract start coordinates [x, y]
    events['start_x'] = events['location'].apply(
        lambda loc: float(loc[0]) if isinstance(loc, (list, tuple)) and len(loc) >= 2 else np.nan
    )
    events['start_y'] = events['location'].apply(
        lambda loc: float(loc[1]) if isinstance(loc, (list, tuple)) and len(loc) >= 2 else np.nan
    )

    # 3. Extract end coordinates [x, y]
    def get_end_x(row):
        if row['type'] == 'Pass' and isinstance(row.get('pass_end_location'), (list, tuple)):
            return float(row['pass_end_location'][0])
        elif row['type'] == 'Carry' and isinstance(row.get('carry_end_location'), (list, tuple)):
            return float(row['carry_end_location'][0])
        elif row['type'] == 'Dribble' and isinstance(row.get('location'), (list, tuple)):
            return float(row['location'][0]) + 3.0 if row.get('dribble_outcome') == 'Complete' else float(row['location'][0])
        return np.nan

    def get_end_y(row):
        if row['type'] == 'Pass' and isinstance(row.get('pass_end_location'), (list, tuple)):
            return float(row['pass_end_location'][1])
        elif row['type'] == 'Carry' and isinstance(row.get('carry_end_location'), (list, tuple)):
            return float(row['carry_end_location'][1])
        elif row['type'] == 'Dribble' and isinstance(row.get('location'), (list, tuple)):
            return float(row['location'][1])
        return np.nan

    events['end_x'] = events.apply(get_end_x, axis=1)
    events['end_y'] = events.apply(get_end_y, axis=1)

    # 4. Drop non-spatial rows missing coordinates
    # events = events.dropna(subset=['start_x', 'start_y', 'end_x', 'end_y']).copy()

    # 5. Calculate spatial deltas and yardage
    events['delta_x'] = events['end_x'] - events['start_x']  # + is forward progression
    events['delta_y'] = events['end_y'] - events['start_y']
    events['distance_yards'] = round(np.sqrt(events['delta_x']**2 + events['delta_y']**2), 2)

    # 6. Progressive Action Flags
    in_opp_box = (events['end_x'] >= 102.0) & (events['end_y'] >= 18.0) & (events['end_y'] <= 62.0)

    events['is_progressive_pass'] = (
        (events['type'] == 'Pass') &
        (events.get('pass_outcome').isna()) &
        ((events['delta_x'] >= 10.0) | in_opp_box)
    )

    events['is_progressive_carry'] = (
        (events['type'] == 'Carry') &
        ((events['delta_x'] >= 10.0) | in_opp_box)
    )

    events['is_progressive_dribble'] = (
        (events['type'] == 'Dribble') &
        (events.get('dribble_outcome') == 'Complete') &
        (events['start_x'] >= 40.0)
    )

    events['is_progressive_action'] = (
        events['is_progressive_pass'] |
        events['is_progressive_carry'] |
        events['is_progressive_dribble']
    )

    # 7. Select ONLY essential columns to compress memory usage footprint
    core_columns = [
        'match_id', 'player', 'team', 'type', 'standardized_timestamp',
        'start_x', 'start_y', 'end_x', 'end_y',
        'delta_x', 'delta_y', 'distance_yards',
        'is_progressive_pass', 'is_progressive_carry',
        'is_progressive_dribble', 'is_progressive_action',
        'defensive_effort_count',
        'avg_pass_length', 'most_frequent_pass_style', 'total_ball_possession_events_per_match'
    ]

    # Retain optional league_name tag if present
    if 'league_name' in events.columns:
        core_columns.append('league_name')

    compressed_df = events[core_columns].copy()

    return compressed_df

In [86]:
# Putting it to Work on the Master DataFrame
df = process_and_compress_spatial_events(original_df)
print("Vector Processing Output Preview:")

df.sample(5)

Vector Processing Output Preview:


,match_id,player,team,type,standardized_timestamp,start_x,start_y,end_x,end_y,delta_x,delta_y,distance_yards,is_progressive_pass,is_progressive_carry,is_progressive_dribble,is_progressive_action,defensive_effort_count,avg_pass_length,most_frequent_pass_style,total_ball_possession_events_per_match,league_name
102092,269714,Alex Oxlade-Chamberlain,Arsenal,Carry,58.93,55.8,71.6,70.7,73.2,14.9,1.6,14.99,False,True,False,True,NaN,NaN,NaN,1.0,Premier League
62877,152728,Ashley Young,Manchester United,Pass,71.06,81.2,67.7,85.1,57.8,3.9,-9.9,10.64,False,False,False,False,NaN,10.640489,Right Foot,NaN,Premier League
92228,241144,Alberto Moreno Pérez,Liverpool,Carry,63.37,85.9,25.7,79.4,27.9,-6.5,2.2,6.86,False,False,False,False,NaN,NaN,NaN,1.0,Premier League
96251,252845,Kolo Habib Touré,Liverpool,Carry,16.64,44.6,19.8,45.6,21.7,1.0,1.9,2.15,False,False,False,False,NaN,NaN,NaN,1.0,Premier League
90403,236070,Mamadou Sakho,Liverpool,Carry,70.97,28.8,17.9,28.8,17.9,0.0,0.0,0.00,False,False,False,False,NaN,NaN,NaN,1.0,Premier League




---



## Machine Learning for Feature Engineering

### Utilizing Linear Regression + Regularization to Create each Defender's `Ball-Carrying Ability` Score

In [103]:
# Removing Columns & Values which Corrode & add Unnecessary Noise to the Dataset
X = df[(df['total_ball_possession_events_per_match'].isna()) & ((df['total_ball_possession_events_per_match'].isnull()))]
X.drop(columns = ['total_ball_possession_events_per_match', 'defensive_effort_count'], inplace = True)

# Type Setting
cols_to_convert = X.select_dtypes(include = ['category', 'object', 'bool']).columns
exclude_cols = ['type', 'team', 'most_frequent_pass_style', 'league_name', 'player']
final_cols_for_conversion = [col for col in cols_to_convert if col not in exclude_cols]

X[final_cols_for_conversion] = X[final_cols_for_conversion].fillna(0).astype(int)

# Further Cleaning/Prep
X = X[(~X['avg_pass_length'].isnull()) & (~X['avg_pass_length'].isna())]

for i in X:
  if i in ['avg_pass_length', 'distance_yards']:
    X[i] = X[i].round(2)

X.sample(6)

,match_id,player,team,type,standardized_timestamp,start_x,start_y,end_x,end_y,delta_x,delta_y,distance_yards,is_progressive_pass,is_progressive_carry,is_progressive_dribble,is_progressive_action,avg_pass_length,most_frequent_pass_style,league_name
47499,113704,Matteo Darmian,Manchester United,Pass,10.71,51.4,79.0,58.4,60.9,7.0,-18.1,19.41,0,0,0,0,19.41,Right Foot,Premier League
47128,112950,Mamadou Sakho,Liverpool,Pass,47.64,34.8,17.4,40.4,40.1,5.6,22.7,23.38,0,0,0,0,23.38,Left Foot,Premier League
10157,22326,Sergino Dest,Barcelona,Pass,15.55,117.4,58.7,111.2,38.8,-6.2,-19.9,20.84,0,0,0,0,20.84,Right Foot,La Liga
50035,119945,Ignacio Monreal Eraso,Arsenal,Pass,36.06,24.7,12.7,64.1,32.8,39.4,20.1,44.23,0,0,0,0,44.23,Right Foot,Premier League
4367,10290,Sergi Roberto Carnicer,Barcelona,Pass,72.29,63.6,80.0,70.2,64.1,6.6,-15.9,17.22,0,0,0,0,17.22,NaN,La Liga
43142,103628,Mamadou Sakho,Liverpool,Pass,75.08,35.6,58.3,48.6,72.9,13.0,14.6,19.55,1,0,0,1,19.55,Left Foot,Premier League


In [105]:
import matplotlib.pyplot as plt
import seaborn as sns

# Identify categorical columns to be one-hot encoded.
# Exclude 'player' and 'match_id' as they are typically identifiers or high-cardinality features.
categorical_cols_to_encode = [col for col in X.select_dtypes(include=['object', 'category']).columns if col not in ['player', 'match_id']]

# Create dummy variables for the selected categorical columns
X_dummies = pd.get_dummies(X[categorical_cols_to_encode], drop_first=True) # drop_first to avoid multicollinearity

# Drop the original categorical columns from X
X_numeric_and_ids = X.drop(columns=categorical_cols_to_encode)

# Concatenate the numerical/id columns with the one-hot encoded columns
X = pd.concat([X_numeric_and_ids, X_dummies], axis=1)

X.sample(7)

,match_id,player,standardized_timestamp,start_x,start_y,end_x,end_y,delta_x,delta_y,distance_yards,is_progressive_pass,is_progressive_carry,is_progressive_dribble,is_progressive_action,avg_pass_length,team_Barcelona,team_Bayern Munich,team_Borussia Dortmund,team_Liverpool,team_Manchester United,team_Real Madrid,most_frequent_pass_style_Left Foot,most_frequent_pass_style_No Touch,most_frequent_pass_style_Other,most_frequent_pass_style_Right Foot,league_name_La Liga,league_name_Premier League
4528,10585,Gerard Piqué Bernabéu,74.03,47.2,76.3,56.7,78.1,9.5,1.8,9.67,0,0,0,0,9.67,True,False,False,False,False,False,False,False,False,True,True,False
49969,119832,Gabriel Armando de Abreu,9.39,114.9,62.6,103.6,60.7,-11.3,-1.9,11.46,0,0,0,0,11.46,False,False,False,False,False,False,True,False,False,False,False,True
7222,16025,Sergi Roberto Carnicer,3.26,55.8,75.0,78.0,75.0,22.2,0.0,22.20,1,0,0,1,22.20,True,False,False,False,False,False,True,False,False,False,True,False
43074,103486,Mamadou Sakho,30.82,66.5,10.7,55.3,34.6,-11.2,23.9,26.39,0,0,0,0,26.39,False,False,False,True,False,False,True,False,False,False,False,True
12162,26903,Gerard Piqué Bernabéu,57.49,35.3,37.9,9.6,37.1,-25.7,-0.8,25.71,0,0,0,0,25.71,True,False,False,False,False,False,False,False,False,True,True,False
54422,131148,Dejan Lovren,47.41,35.8,37.2,55.9,73.7,20.1,36.5,41.67,1,0,0,1,41.67,False,False,False,True,False,False,False,False,False,True,False,True
4669,10846,Sergi Roberto Carnicer,43.61,70.2,60.4,83.0,74.7,12.8,14.3,19.19,0,0,0,0,19.19,True,False,False,False,False,False,False,False,False,True,True,False


In [108]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Define target variable (y) and features (X_model)
# We will predict 'is_progressive_action'
y = X['is_progressive_action']

# Drop columns that are identifiers or the target itself from the feature set
X_model = X.drop(columns=[
    'match_id',
    'player',
    'is_progressive_action',
    'start_x', 'start_y', 'end_x', 'end_y', 'delta_x', 'delta_y' # These are raw spatial data, distance_yards is a derived feature
])

# For simplicity, we will impute remaining NaNs with the mean of their respective columns.
# This is crucial before training machine learning models.
for col in X_model.columns:
    if X_model[col].isnull().any():
        if pd.api.types.is_numeric_dtype(X_model[col]):
            X_model[col] = X_model[col].fillna(X_model[col].mean())
        else:
            # For non-numeric columns that somehow slipped through (shouldn't happen after get_dummies)
            X_model[col] = X_model[col].fillna(X_model[col].mode()[0])

# Convert boolean columns to int (0 or 1) for the model
for col in X_model.select_dtypes(include='bool').columns:
    X_model[col] = X_model[col].astype(int)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_model, y, test_size=0.3, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (28006, 18)
Shape of y_train: (28006,)
Shape of X_test: (12003, 18)
Shape of y_test: (12003,)


In [109]:
# Initialize and train a RandomForestClassifier
# Using class_weight='balanced' to handle potential imbalance in progressive vs non-progressive actions
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict probabilities on the test set
y_pred_proba = rf_model.predict_proba(X_test)[:, 1] # Probability of the positive class (is_progressive_action = True)

# Evaluate the model
print("\nRandom Forest Classifier Performance on Test Set:")
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int)))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Get feature importances
feature_importances = pd.Series(rf_model.feature_importances_, index=X_model.columns).sort_values(ascending=False)
print("\nTop 10 Feature Importances:")
display(feature_importances.head(10))


Random Forest Classifier Performance on Test Set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8945
           1       1.00      1.00      1.00      3058

    accuracy                           1.00     12003
   macro avg       1.00      1.00      1.00     12003
weighted avg       1.00      1.00      1.00     12003

ROC AUC Score: 1.0000

Top 10 Feature Importances:


,0
is_progressive_pass,0.913278
avg_pass_length,0.041468
distance_yards,0.036693
standardized_timestamp,0.004528
team_Barcelona,0.000853
most_frequent_pass_style_Left Foot,0.000712
league_name_La Liga,0.000532
most_frequent_pass_style_Right Foot,0.000484
league_name_Premier League,0.000431
team_Manchester United,0.000383


Now, let's generate a 'progressive action score' for each player. We'll use the trained Random Forest model to predict the probability of `is_progressive_action` for *all* relevant events in the original `X` DataFrame (not just the test set), and then average these probabilities per player.

In [112]:
# Predict probabilities for the entire dataset
# Ensure all columns in X_model are present and in the same order as during training

# Recreate X_predict, dropping identifiers and target, and handling NaNs consistently
X_predict = X.drop(columns=[
    'match_id',
    'player',
    'is_progressive_action',
    'start_x', 'start_y', 'end_x', 'end_y', 'delta_x', 'delta_y' # 'type' column is already one-hot encoded and removed from X. It should not be dropped again.
])

for col in X_predict.columns:
    if X_predict[col].isnull().any():
        if pd.api.types.is_numeric_dtype(X_predict[col]):
            X_predict[col] = X_predict[col].fillna(X_predict[col].mean())
        else:
            X_predict[col] = X_predict[col].fillna(X_predict[col].mode()[0])

for col in X_predict.select_dtypes(include='bool').columns:
    X_predict[col] = X_predict[col].astype(int)

# Make sure the columns of X_predict match X_model (used for training)
X_predict = X_predict[X_model.columns]

player_action_probabilities = rf_model.predict_proba(X_predict)[:, 1]

# Create a DataFrame to store player information and their predicted probabilities
# 'match_id' and 'player' are still in the current 'X' DataFrame.
# 'team' was removed from 'X' during one-hot encoding. Retrieve it from the 'df' DataFrame
# which still holds the original 'team' column and has matching indices.
player_scores_df = pd.DataFrame({
    'match_id': X['match_id'],
    'player': X['player'],
    'team': df.loc[X.index, 'team'],
    'progressive_action_probability': player_action_probabilities
})

# Aggregate by player to get an average 'progressive action score'
# We'll also count the number of events considered for each player's score
player_overall_scores = player_scores_df.groupby(['player', 'team']).agg(
    mean_progressive_action_score=('progressive_action_probability', 'mean'),
    num_events=('progressive_action_probability', 'count')
).reset_index().sort_values(by='mean_progressive_action_score', ascending=False)

display(player_overall_scores.head(10))

print("\nInterpretation: A higher 'mean_progressive_action_score' indicates a player is more likely to perform a progressive action based on their characteristics and context, according to the Random Forest model. 'num_events' shows how many events were included in their score calculation.")

,player,team,mean_progressive_action_score,num_events
78,Thomas Meunier,Borussia Dortmund,0.666667,3
2,Alex Oxlade-Chamberlain,Arsenal,0.527059,17
5,Antonios Papadopoulos,Borussia Dortmund,0.492000,10
37,Jordon Ibe,Liverpool,0.427143,7
70,Raphaël Varane,Real Madrid,0.404915,59
49,Mamadou Sakho,Liverpool,0.391801,1244
77,Sergio Ramos García,Real Madrid,0.391724,58
14,Daley Blind,Manchester United,0.389458,2215
19,Eric Dier,Bayern Munich,0.384370,119
56,Michael Carrick,Manchester United,0.378621,29



Interpretation: A higher 'mean_progressive_action_score' indicates a player is more likely to perform a progressive action based on their characteristics and context, according to the Random Forest model. 'num_events' shows how many events were included in their score calculation.


In [114]:
# Saving the ML-Modeled Dataset as well as the Transformed one from Before

df.to_csv('processed_df.csv')
player_overall_scores.to_csv('player_model_scores.csv')